# [문제 1] Fashion MNIST 데이터 정규화를 위한 Mean과 Std. 값 찾기

In [14]:
import torch
from torch.utils.data import random_split, DataLoader
from torchvision import datasets, transforms
from wandb.util import download_file_into_memory

In [25]:
data_path = "."
f_mnist_train = datasets.FashionMNIST(data_path, train=True, download=True, transform=transforms.ToTensor())
f_mnist_train, f_mnist_validation = random_split(f_mnist_train, [55_000, 5_000])

images = torch.stack([img for img, _ in f_mnist_train], dim=0)

mean = images.mean()
std = images.std()

print(f"Mean: {mean.item():.4f}")
print(f"Std: {std.item():.4f}")

Mean: 0.2860
Std: 0.3530


# [문제 2] Fashion MNIST 데이터에 대하여 CNN 학습시키기

In [ ]:
!git clone https://github.com/wjm0423/link_dl.git

In [ ]:
!pip install wandb

In [ ]:
import wandb
wandb.login()

In [ ]:
!pip install torchinfo

In [ ]:
import torch
from torch import nn, optim
from datetime import datetime
import os
from pathlib import Path
from torch.optim import lr_scheduler

try:
  BASE_PATH = str(Path(__file__).resolve().parent.parent.parent)  # BASE_PATH: /Users/yhhan/git/link_dl
except NameError:
  BASE_PATH = str(Path.cwd() / 'link_dl')
import sys
sys.path.append(BASE_PATH)
os.listdir(BASE_PATH)

try:
  CURRENT_FILE_PATH = os.path.dirname(os.path.abspath(__file__))
except NameError:
  CURRENT_FILE_PATH = str(Path.cwd())
CHECKPOINT_FILE_PATH = os.path.join(CURRENT_FILE_PATH, "checkpoints")
os.makedirs(CHECKPOINT_FILE_PATH, exist_ok=True)
if not os.path.isdir(CHECKPOINT_FILE_PATH):
  os.makedirs(CHECKPOINT_FILE_PATH)

import sys
sys.path.append(BASE_PATH)

from _01_code._09_fcn_best_practice.c_trainer import ClassificationTrainer
from _03_homeworks.homework_3.a_fashion_mnist_data import get_fashion_mnist_data
from _01_code._16_modern_cnns.a_arg_parser import get_parser


def get_resnet_model(num_classes=10):
    class ResnetBlock(nn.Module):

      def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()

        # ------------------------------------
        # ResNet 블록: H(x) = x + F(x)의 F(x)
        # ------------------------------------
        # ResNet 블록의 첫 번째 Convolution Layer
        # 입력 피처 맵(in_channels)을 받아 out_channels개의 3 × 3 필터를 학습하여 출력 피처 맵을 생성
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False
        )
        # 배치 정규화: 각 피처에 대해 배치 단위로 평균과 분산을 계산해 정규화
        self.bn1 = nn.BatchNorm2d(out_channels)
        # 활성화 함수-ReLU: 입력값 x가 0 이하면 0, 0 초과면 x 반환
        self.relu = nn.ReLU(inplace=True)

        # ResNet 블록의 첫 번째 Convolution Layer 이후 Convolution Layer
        # 블록 내부에서 필터링만 수행
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # H(x) = x + F(x)에서 x와 F(x)의 크기가 다를 때 x(=identity)에 적용하는 다운샘플링
        # stride가 1이 아닐 때 크기 차이가 발생하기 때문에 stride != 1 or self.in_channels != out_channels로 조건을 준다.
        self.downsample = downsample

      # ResNet 블록의 순전파 순서
      def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # stride != 1 or self.in_channels != out_channels이면 identity에 다운샘플링을 적용하여 out과 크기를 맞춤.
        if self.downsample is not None:
          identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

    # ------------------------------------
    # ResNet-18
    # ------------------------------------
    class ResNet18(nn.Module):
      def __init__(self):
        super().__init__()

        # 처음 stem 부분 → Conv 사용
        # 입력: Fashion-MNIST 이미지 → 1 × 28 × 28
        # Fashion-MNIST 이미지는 흑백 이미지이기 때문에 RGB값이 없어 채널 값은 1
        # B × 1 × 28 × 28 --> B × 64 × {(28 - 3 + 2) / 1 + 1} × {(28 - 3 + 2) / 1 + 1} = B × 64 × 28 × 28
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=64, kernel_size=(3, 3), stride=(1, 1), padding=1, bias=False),
            nn.BatchNorm2d(num_features=64, eps=1e-05, momentum=0.1),
            nn.ReLU(inplace=True)
        )

        self.in_channels = 64

        # ResNet stages (2 blocks × 4 layers): 각 layer는 ResNetBlock을 2개씩 쌓은 것.
        # B × 64 × 28 × 28 --> B × 64 × {(28 - 3 + 2) / 1 + 1} × {(28 - 3 + 2) / 1 + 1} = B × 64 × 28 × 28
        self.layer1 = self._make_layer(out_channels=64, blocks=2, stride=1)
        # B × 64 × 28 ×28 --> B × 128 × {(28 - 3 + 2) / 2 + 1} × {(28 - 3 + 2) / 2 + 1} = B × 128 × 14 × 14
        self.layer2 = self._make_layer(out_channels=128, blocks=2, stride=2)
        # B × 128 × 28 ×28 --> B × 256 × {(14 - 3 + 2) / 2 + 1} × {(14 - 3 + 2) / 2 + 1} = B × 256 × 7 × 7
        self.layer3 = self._make_layer(out_channels=256, blocks=2, stride=2)
        # B × 256 × 7 ×7 --> B × 512 × {(7 - 3 + 2) / 2 + 1} × {(7 - 3 + 2) / 2 + 1} = B × 512 × 4 × 4
        self.layer4 = self._make_layer(out_channels=512, blocks=2, stride=2)

        # B × 512 × 4 × 4 --> B × 512 × 1 × 1
        # 각 채널에 대해 모든 4 × 4 공간 차원의 평균값을 계산하여 1 × 1 크기의 텐서로 만듦. 즉, 각 샘플을 512개의 특징 벡터로 요약.
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.4) # # p: dropout probability
        # B × 512 --> B × 10
        # 512개의 특징 벡터를 가지고 판단하여 샘플이 10개(num_classes)의 클래스별로 분류될 확률을 계산
        self.fc = nn.Linear(512, num_classes)

      def _make_layer(self, out_channels, blocks, stride):
        # ResNet 블록들을 쌓아 하나의 layer를 생성
        downsample = None

        # 필요한 경우 identity 다운샘플링
        # downsample 로직 → Conv/BatchNorm 활용
        if stride != 1 or self.in_channels != out_channels:
          downsample = nn.Sequential(
              nn.Conv2d(self.in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
              nn.BatchNorm2d(out_channels)
        )

        # 첫 번째 ResNet 블록을 만들어 layers에 추가
        layers = []
        layers.append(ResnetBlock(self.in_channels, out_channels, stride=stride, downsample=downsample))

        self.in_channels = out_channels

        # 나머지 ResNet 블록을 만들어 layers에 추가하여 반환
        for _ in range(1, blocks):
          layers.append(ResnetBlock(out_channels, out_channels))

        return nn.Sequential(*layers)

      def forward(self, x):
        """
        Pass the input through the net.
        """
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        # B × 512 × 1 × 1 --> B × 512
        x = self.dropout(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

    # ResNet18 인스턴스를 만들어 반환
    my_model = ResNet18()

    return my_model


def main(args):
  # 프로젝트 이름이나 체크포인트에 사용하기 위해 현재 시간을 타임스탬프로 생성
  run_time_str = datetime.now().astimezone().strftime('%Y-%m-%d_%H-%M-%S')

  # 학습 설정
  config = {
      'epochs': args.epochs,
      'batch_size': args.batch_size,
      'validation_intervals': args.validation_intervals,
      'learning_rate': args.learning_rate,
      'early_stop_patience': args.early_stop_patience,
      'early_stop_delta': args.early_stop_delta,
      'weight_decay': args.weight_decay
  }

  # Wandb에 프로젝트 로깅하여 출력
  project_name = "cnn_fashion_mnist"
  name = "resnet_{0}".format(run_time_str)
  wandb.init(
      mode="online" if args.wandb else "disabled",
      project=project_name,
      notes="fashion mnist experiment with resnet",
      tags=["resnet", "fashion_mnist"],
      name=name,
      config=config
  )
  print(args)
  print(wandb.config)

  # CUDA 가능 여부에 따라 GPU나 CPU 선택
  device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
  print(f"Training on device {device}.")

  # 학습용, 검증용 DataLoader 및 Data Augmentation을 적용한 이미지 transform을 호출
  train_data_loader, validation_data_loader, fashion_mnist_transforms = get_fashion_mnist_data()
  model = get_resnet_model(num_classes=10)
  model.to(device)

  # ResNet18 모델 생성 후 GPU/CPU 디바이스로 이동
  from torchinfo import summary
  summary(model=model,
          input_size=(1, 1, 28, 28),
          col_names=["kernel_size", "input_size", "output_size", "num_params", "mult_adds"]
  )

  # 옵티마이저는 Adam를 사용
  optimizer = optim.Adam(
      model.parameters(),
      lr=wandb.config.learning_rate,
      weight_decay=config['weight_decay']
  )

  # 학습률 스케줄러
  scheduler = lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

  # Fashion-MNIST 데이터셋 분류 훈련용 객체 생성.
  # train_loop()로 훈련, 검증, early stopping, 체크포인트 저장, wandb 로깅 등을 수행
  classification_trainer = ClassificationTrainer(
      project_name, model, optimizer, train_data_loader, validation_data_loader, fashion_mnist_transforms,
      run_time_str, wandb, device, CHECKPOINT_FILE_PATH, scheduler=scheduler
  )
  classification_trainer.train_loop()

  # wandb 세션 종료
  wandb.finish()

if __name__ == "__main__":
  # parser = get_parser()
  # args = parser.parse_args()
  # python _01_code/_09_modern_cnns/_02_googlenet/a_cifar10_train_googlenet.py --wandb -v 10
  from types import SimpleNamespace

  args = SimpleNamespace(
      wandb=True,
      batch_size=512,
      epochs=200,
      learning_rate=0.0002,
      validation_intervals=10,
      early_stop_patience=15,
      early_stop_delta=1e-05,
      weight_decay=0.0005
  )

  main(args)

# [문제 3] 학습 완료된 모델로 테스트 데이터 Accuracy	확인하기

In [ ]:
import numpy as np
import torch
import os

from matplotlib import pyplot as plt
from pathlib import Path

try:
    BASE_PATH = str(Path(__file__).resolve().parent.parent.parent)  # BASE_PATH: /Users/yhhan/git/link_dl
except NameError:
    BASE_PATH = str(Path.cwd() / 'link_dl')
try:
    CURRENT_FILE_PATH = os.path.dirname(os.path.abspath(__file__))
except NameError:
    CURRENT_FILE_PATH = str(Path.cwd())
CHECKPOINT_FILE_PATH = os.path.join(CURRENT_FILE_PATH, "checkpoints")
os.makedirs(CHECKPOINT_FILE_PATH, exist_ok=True)
if not os.path.isdir(CHECKPOINT_FILE_PATH):
  os.makedirs(CHECKPOINT_FILE_PATH)

import sys
sys.path.append(BASE_PATH)

from _01_code._09_fcn_best_practice.d_tester import ClassificationTester
from _03_homeworks.homework_3.a_fashion_mnist_data import get_fashion_mnist_test_data

import torchvision

USE_PYTORCH_MODEL = False

def main():
  f_mnist_test_images, test_data_loader, f_mnist_transforms = get_fashion_mnist_test_data()

  test_model = torchvision.models.resnet18(num_classes=10) if USE_PYTORCH_MODEL else get_resnet_model(num_classes=10)

  project_name = "cnn_fashion_mnist"
  classification_tester = ClassificationTester(
    project_name, test_model, test_data_loader, f_mnist_transforms, CHECKPOINT_FILE_PATH
  )
  classification_tester.test()

  print()

  img, label = f_mnist_test_images[0]
  print("     LABEL:", label)
  plt.imshow(img)
  plt.show()

  output = classification_tester.test_single(
    f_mnist_test_images[0][0]
  )
  print("PREDICTION:", output)


if __name__ == "__main__":
  main()

# [문제 4] 샘플 테스트 데이터 분류 예측 결과 확인하기